In [ ]:
# =======================================================================
#                          ECC Residuals Heatmaps  
# -----------------------------------------------------------------------
# - Works with 8-bit and 16-bit images
# - Blue = low residual (good alignment), Red = high residual (mismatch)
# =======================================================================

import cv2
import numpy as np
from pathlib import Path
import glob
import os

# ----------------- USER SETTINGS -----------------
input_folder      = "input folder"   # <------ Update
pattern           = "*.tif"          # <------ Change file type for use images (THIS CODE WILL NOT WORK IF FILE TYPES FOR IMAGES DO NOT MATCH)
output_folder     = "output folder"  # <------ Update
reference_index   = 0
warp_mode         = cv2.MOTION_AFFINE
pyramid_levels    = 3
max_iters         = 300
epsilon           = 1e-6
preblur_ksize     = (5, 5)
colormap          = cv2.COLORMAP_JET
tick_count        = 6

# --- Percentile cutoff for scaling ---
global_percentile = 99.5   # e.g., 99.5% of residuals, ignoring extreme outliers
# -------------------------------------------------

Path(output_folder).mkdir(parents=True, exist_ok=True)

# ---------- Utility Functions ----------
def imread_gray_unchanged(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    if img.ndim == 3:  # BGR->Gray (keep bit depth)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img

def to_float01(img):
    if img.dtype == np.uint8:
        base = 255.0
    elif img.dtype == np.uint16:
        base = 65535.0
    else:
        base = float(img.max()) if img.max() > 0 else 1.0
    return img.astype(np.float32) / base, base

def build_pyr(img_f01, L):
    pyr = [img_f01]
    for _ in range(L - 1):
        pyr.append(cv2.pyrDown(pyr[-1]))
    return pyr[::-1]  # coarse -> fine

def scale_warp_between_levels(W, scale, mode):
    W = W.copy()
    if mode in (cv2.MOTION_TRANSLATION, cv2.MOTION_EUCLIDEAN, cv2.MOTION_AFFINE):
        W[0, 2] *= scale
        W[1, 2] *= scale
    else:  # homography
        W[0, 2] *= scale
        W[1, 2] *= scale
    return W

def ecc_multilevel(ref_f01, tgt_f01, L, mode, criteria, blur_ksize):
    ref_p = build_pyr(ref_f01, L)
    tgt_p = build_pyr(tgt_f01, L)
    W = np.eye(3, dtype=np.float32) if mode == cv2.MOTION_HOMOGRAPHY else np.eye(2, 3, dtype=np.float32)

    for li, (r, t) in enumerate(zip(ref_p, tgt_p)):
        h, w = r.shape
        if t.shape != (h, w):
            t = cv2.resize(t, (w, h), interpolation=cv2.INTER_LINEAR)

        r_blur = cv2.GaussianBlur(r, blur_ksize, 0) if blur_ksize else r
        t_blur = cv2.GaussianBlur(t, blur_ksize, 0) if blur_ksize else t

        try:
            _, W = cv2.findTransformECC(
                r_blur, t_blur, W, mode,
                criteria=criteria, inputMask=None, gaussFiltSize=5
            )
        except cv2.error:
            pass

        if li < L - 1:  # scale translation for next finer level
            W = scale_warp_between_levels(W, 2.0, mode)
    return W

def warp_like_ref(img, W, mode, w, h):
    if mode == cv2.MOTION_HOMOGRAPHY:
        return cv2.warpPerspective(img, W, (w, h), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
    else:
        return cv2.warpAffine(img, W, (w, h), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)

def make_colorbar(height, hi_norm, base_range, cm, tick_count=6, percentile=99.5):
    # Colorbar gradient: bottom = 0 (blue), top = 1 (red)
    grad = np.linspace(1, 0, height, dtype=np.float32)[:, None]
    grad_u8 = (grad * 255).astype(np.uint8)
    bar = cv2.applyColorMap(grad_u8, cm)
    bar = cv2.resize(bar, (40, height), interpolation=cv2.INTER_NEAREST)

    max_val_counts = hi_norm * base_range
    font = cv2.FONT_HERSHEY_DUPLEX

    top_label = f"{max_val_counts:.1f}"
    (top_w, _), _ = cv2.getTextSize(top_label, font, 0.55, 1)
    tick_pad_w = max(80, top_w + 25)

    # Title with percentile info
    title = (
        f"Residual |ref - warped| Gray Level Difference\n"
        f"(Top = {percentile}th percentile ≈ {max_val_counts:.1f})"
    )
    (tw, th), _ = cv2.getTextSize(title, font, 0.6, 1)
    text_canvas = np.full((th + 20, tw + 20, 3), 255, np.uint8)
    cv2.putText(text_canvas, title, (10, th + 5), font, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    text_rot = cv2.rotate(text_canvas, cv2.ROTATE_90_COUNTERCLOCKWISE)
    tH, tW = text_rot.shape[:2]
    title_pad_w = tW + 20

    pad = np.full((height, tick_pad_w + title_pad_w, 3), 255, np.uint8)
    cb = np.concatenate([bar, pad], axis=1)

    # Tick marks
    ticks = np.linspace(0.0, 1.0, tick_count)
    for t in ticks:
        y = int(round((1.0 - t) * (height - 1)))
        x0 = bar.shape[1]
        cv2.line(cb, (x0, y), (x0 + 6, y), (0, 0, 0), 1)
        val = t * max_val_counts
        label = f"{val:.1f}"
        y_lbl = max(20, y + 12) if t == 1.0 else y + 4
        cv2.putText(cb, label, (x0 + 12, y_lbl), font, 0.55, (0, 0, 0), 1, cv2.LINE_AA)

    # Bottom label for clarity
    cv2.putText(cb, "0 = perfect alignment", (bar.shape[1] + 12, height - 10),
                font, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    # Place rotated title
    x_title0 = bar.shape[1] + tick_pad_w + (title_pad_w - tW) // 2
    y_title0 = (cb.shape[0] - tH) // 2
    cb[y_title0:y_title0+tH, x_title0:x_title0+tW] = text_rot

    return cb

def save_with_colorbar(heat_bgr, hi_norm, base_range, out_path, percentile):
    h, w = heat_bgr.shape[:2]
    cb = make_colorbar(h, hi_norm, base_range, colormap, tick_count, percentile)
    sep = np.full((h, 8, 3), 255, np.uint8)
    combo = np.concatenate([heat_bgr, sep, cb], axis=1)
    legend = "Blue = good alignment; Red = higher mismatch."
    cv2.rectangle(combo, (0, h-22), (combo.shape[1], h), (255,255,255), -1)
    cv2.putText(combo, legend, (10, h-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
    cv2.imwrite(str(out_path), combo)

# ---------- Main ----------
def main():
    files = sorted(glob.glob(os.path.join(input_folder, pattern)))
    if not files:
        raise RuntimeError("No images found.")

    ref0 = imread_gray_unchanged(files[reference_index])
    ref0_f01, base_range = to_float01(ref0)
    H, W = ref0.shape
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, max_iters, epsilon)

    # --- First pass: compute percentile cutoff across all residuals ---
    all_residuals = []
    for i, fpath in enumerate(files):
        if i == reference_index:
            continue
        img = imread_gray_unchanged(fpath)
        if img.shape != (H, W):
            img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)
        img_f01, _ = to_float01(img)

        W_est = ecc_multilevel(ref0_f01, img_f01, pyramid_levels, warp_mode, criteria, preblur_ksize)
        warped = warp_like_ref(img_f01, W_est, warp_mode, W, H)
        res = np.abs(ref0_f01 - warped)
        all_residuals.append(res.flatten())

    all_residuals = np.concatenate(all_residuals)
    global_max = np.percentile(all_residuals, global_percentile)

    if global_max <= 0:
        global_max = 1e-6

    print(f"[INFO] Global cutoff (percentile {global_percentile}) = {global_max:.6f} (normalized units)")
    print(f"[INFO] In gray levels = {global_max * base_range:.2f}")

    # --- Second pass: generate heatmaps ---
    for i, fpath in enumerate(files):
        img = imread_gray_unchanged(fpath)
        if img.shape != (H, W):
            img = cv2.resize(img, (W, H), interpolation=cv2.INTER_LINEAR)
        img_f01, _ = to_float01(img)

        if i == reference_index:
            zero_heat = np.zeros((H, W, 3), np.uint8)
            save_with_colorbar(zero_heat, global_max, base_range, Path(output_folder, f"{i:05d}_heatmap.png"), global_percentile)
            continue

        W_est = ecc_multilevel(ref0_f01, img_f01, pyramid_levels, warp_mode, criteria, preblur_ksize)
        warped = warp_like_ref(img_f01, W_est, warp_mode, W, H)
        res = np.abs(ref0_f01 - warped)

        # Normalize to global_max
        res_norm = np.clip(res / global_max, 0.0, 1.0)
        heat = (res_norm * 255.0).astype(np.uint8)

        heat_bgr = cv2.applyColorMap(heat, colormap)
        save_with_colorbar(heat_bgr, global_max, base_range, Path(output_folder, f"{i:05d}_heatmap.png"), global_percentile)

    print(f"Done. Heatmaps saved to {output_folder}")

if __name__ == "__main__":
    main()
